In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler, LabelEncoder

DATA_PATH = "/Users/kasimozel/Desktop/Akıllı Tarım Asistanı/akilli-tarim-karar-sistemi/ml_service/data/"
MODEL_PATH = "/Users/kasimozel/Desktop/Akıllı Tarım Asistanı/akilli-tarim-karar-sistemi/ml_service/models/"

# Veriyi yukle
hava_df = pd.read_csv(DATA_PATH + "turkiye_hava_verisi.csv")

print(f"Toplam satir: {len(hava_df)}")
print(f"Sutunlar: {hava_df.columns.tolist()}")
print(f"\nEksik deger:")
print(hava_df.isnull().sum())

Toplam satir: 21920
Sutunlar: ['tarih', 'max_sicaklik', 'min_sicaklik', 'yagis', 'ruzgar_hizi', 'nem', 'sehir', 'bolge']

Eksik deger:
tarih           0
max_sicaklik    0
min_sicaklik    0
yagis           0
ruzgar_hizi     0
nem             0
sehir           0
bolge           0
dtype: int64


In [2]:
# Bugday icin etiket olustur
def bugday_etiketi(satir):
    max_s  = satir["max_sicaklik"]
    min_s  = satir["min_sicaklik"]
    yagis  = satir["yagis"]
    ruzgar = satir["ruzgar_hizi"]
    nem    = satir["nem"]

    # Uygun degil kosullari
    if min_s < -15:       return "uygun_degil"
    if max_s > 38:        return "uygun_degil"
    if yagis > 50:        return "uygun_degil"
    if ruzgar > 60:       return "uygun_degil"
    if nem > 95:          return "uygun_degil"

    # Riskli kosullar
    if min_s < -5:        return "riskli"
    if max_s > 32:        return "riskli"
    if yagis > 25:        return "riskli"
    if ruzgar > 40:       return "riskli"
    if nem > 85:          return "riskli"

    return "uygun"

hava_df["etiket"] = hava_df.apply(bugday_etiketi, axis=1)

print("Etiket dagilimi:")
print(hava_df["etiket"].value_counts())
print(f"\nYuzdelik:")
print(hava_df["etiket"].value_counts(normalize=True).round(2) * 100)

# Normalizasyon
scaler = StandardScaler()
sayisal_sutunlar = ["max_sicaklik", "min_sicaklik", "yagis", "ruzgar_hizi", "nem"]

hava_df_normalized = hava_df.copy()
hava_df_normalized[sayisal_sutunlar] = scaler.fit_transform(hava_df[sayisal_sutunlar])

# Etiketi sayiya cevir
le = LabelEncoder()
hava_df["etiket_kod"] = le.fit_transform(hava_df["etiket"])
hava_df_normalized["etiket_kod"] = hava_df["etiket_kod"]

print("\nEtiket kodlari:")
for i, sinif in enumerate(le.classes_):
    print(f"  {i} → {sinif}")

# Kaydet
hava_df.to_csv(DATA_PATH + "islenmis_veri.csv", index=False)
hava_df_normalized.to_csv(DATA_PATH + "normalize_veri.csv", index=False)
joblib.dump(scaler, MODEL_PATH + "scaler.pkl")
joblib.dump(le, MODEL_PATH + "label_encoder.pkl")

print("\nKaydedildi!")

Etiket dagilimi:
etiket
riskli         10124
uygun           6534
uygun_degil     5262
Name: count, dtype: int64

Yuzdelik:
etiket
riskli         46.0
uygun          30.0
uygun_degil    24.0
Name: proportion, dtype: float64

Etiket kodlari:
  0 → riskli
  1 → uygun
  2 → uygun_degil

Kaydedildi!


In [3]:
def bugday_etiketi(satir):
    max_s  = satir["max_sicaklik"]
    min_s  = satir["min_sicaklik"]
    yagis  = satir["yagis"]
    ruzgar = satir["ruzgar_hizi"]
    nem    = satir["nem"]

    # Uygun degil kosullari
    if min_s < -15:       return "uygun_degil"
    if max_s > 38:        return "uygun_degil"
    if yagis > 50:        return "uygun_degil"
    if ruzgar > 60:       return "uygun_degil"
    if nem > 98:          return "uygun_degil"  # 95'ten 98'e cikardik

    # Riskli kosullar
    if min_s < -5:        return "riskli"
    if max_s > 32:        return "riskli"
    if yagis > 25:        return "riskli"
    if ruzgar > 40:       return "riskli"
    if nem > 90:          return "riskli"       # 85'ten 90'a cikardik

    return "uygun"

hava_df["etiket"] = hava_df.apply(bugday_etiketi, axis=1)

print("Yeni etiket dagilimi:")
print(hava_df["etiket"].value_counts())
print(f"\nYuzdelik:")
print(hava_df["etiket"].value_counts(normalize=True).round(2) * 100)

Yeni etiket dagilimi:
etiket
riskli         9940
uygun          9340
uygun_degil    2640
Name: count, dtype: int64

Yuzdelik:
etiket
riskli         45.0
uygun          43.0
uygun_degil    12.0
Name: proportion, dtype: float64


In [4]:
# Normalizasyon
scaler = StandardScaler()
sayisal_sutunlar = ["max_sicaklik", "min_sicaklik", "yagis", "ruzgar_hizi", "nem"]

hava_df_normalized = hava_df.copy()
hava_df_normalized[sayisal_sutunlar] = scaler.fit_transform(hava_df[sayisal_sutunlar])

le = LabelEncoder()
hava_df["etiket_kod"] = le.fit_transform(hava_df["etiket"])
hava_df_normalized["etiket_kod"] = hava_df["etiket_kod"]

# Kaydet
hava_df.to_csv(DATA_PATH + "islenmis_veri.csv", index=False)
hava_df_normalized.to_csv(DATA_PATH + "normalize_veri.csv", index=False)
joblib.dump(scaler, MODEL_PATH + "scaler.pkl")
joblib.dump(le, MODEL_PATH + "label_encoder.pkl")

print("Kaydedildi!")
print(f"  → islenmis_veri.csv")
print(f"  → normalize_veri.csv")
print(f"  → scaler.pkl")
print(f"  → label_encoder.pkl")

Kaydedildi!
  → islenmis_veri.csv
  → normalize_veri.csv
  → scaler.pkl
  → label_encoder.pkl
